## 10. 自定义工具：进程内 MCP Server

> 来源：[Give Claude custom tools](https://code.claude.com/docs/en/agent-sdk/custom-tools)

让 Claude 调你自己的本地能力（业务函数、数据库封装、配置中心），是 Agent SDK 真正产生工程价值的地方。做法是把 Python 函数包成**进程内 SDK MCP Server**——不用单独起外部服务，工具和你的应用同进程。


### 10.1 三步接入

1. `@tool(name, description, input_schema)` 把 async 函数声明成工具，handler 返回 `{"content": [{"type": "text", "text": ...}]}`。装饰器返回的是一个工具对象（打包了 name、schema 和 handler），后续注册传的就是这个对象。
2. `create_sdk_mcp_server(name, version, tools=[...])` 打包成 server 对象——`tools` 列表里写函数名是在传对象引用，不是传名字字符串。
3. 挂到 `mcp_servers={...}`，并把工具全名加进 `allowed_tools` **免审批**——不在名单里的工具不是不能调，是每次调用都走权限流程（§9.1「权限评估顺序」）；headless 场景没配任何审批处理时，表现就是"看得到但用不了"。

工具全名规则：`mcp__<server名>__<tool名>`。两段名字的来源都有指定位置，别处的名字不参与拼接：

- **tool 名**取 `@tool` 的第一个参数，handler 的函数名与命名无关——`@tool("greet", ...)` 装饰的函数叫 `greet_user`，全名是 `mcp__demo__greet`，不是 `mcp__demo__greet_user`。
- **server 名**取 `mcp_servers` dict 的 key，`create_sdk_mcp_server` 的 `name` 参数不进全名，两处不一致时以 dict key 为准：

```python
weather_server = create_sdk_mcp_server(
    name="weather-utils",       # 这个 name 不进全名
    version="1.0.0",
    tools=[get_temperature],    # 传对象引用；注册名以 @tool("get_temperature", ...) 为准
)

options = ClaudeAgentOptions(
    mcp_servers={"weather": weather_server},          # dict key "weather" 才是全名里的 server 名
    allowed_tools=["mcp__weather__get_temperature"],  # mcp__<dict key>__<tool 名>
    # allowed_tools=["mcp__weather__*"],              # 通配符：该 server 的全部工具
)
```

成本提醒：挂上的每个工具定义每轮都占 context——工具多了见 §12.4 的 tool search。

### 10.2 Schema 的两档写法

- **简易 dict**：`{"latitude": float}`，SDK 自动转 JSON Schema。**每个 key 都视为必填**；要可选参数就不写进 schema，在 description 里说明、handler 里用 `args.get("hours", 12)` 读。
- **完整 JSON Schema dict**：`@tool` 直接接受。简易 dict 表达不了 `enum`、取值范围、嵌套对象，需要这些约束就必须切到这一档（官方单位换算器示例的 schema）：

```python
@tool(
    "convert_units",
    "Convert a value from one unit to another",
    {
        "type": "object",
        "properties": {
            "unit_type": {
                "type": "string",
                "enum": ["length", "temperature", "weight"],  # 简易 dict 写不出 enum
                "description": "Category of unit",
            },
            "value": {"type": "number", "description": "Value to convert"},
        },
        "required": ["unit_type", "value"],
    },
)
```


### 10.3 Annotations 与并行

annotation 是工具定义上附带的一段**行为性质自述**（概念来自 MCP 协议的 tool annotations）：工具作者在注册工具时声明"这个工具只读吗、有破坏性吗、幂等吗、会触达外部系统吗"，随工具定义一起进 context，供 Claude 做调度决策时参考。`input_schema` 回答"怎么调用"，annotation 回答"调用了会怎样"——两者互补，都是工具定义的一部分。

写法是 `@tool(..., annotations=ToolAnnotations(readOnlyHint=True))`，四个 hint 全是布尔值：

| Hint | 默认值 | 含义 |
|---|---|---|
| `readOnlyHint` | `False` | 不修改环境。**唯一有行为影响的 hint**——标了它，Claude 才会把这个工具与其他只读工具并行调用 |
| `destructiveHint` | `True` | 可能做破坏性更新，仅供参考 |
| `idempotentHint` | `False` | 同参数重复调用无额外效果，仅供参考 |
| `openWorldHint` | `True` | 会触达进程外部系统，仅供参考 |

两个易错点：

- 默认值偏"危险侧"——`destructiveHint`/`openWorldHint` 默认就是 `True`，不标注不等于声明了"安全"。
- **annotation 是元数据不是约束**——SDK 不校验声明与实现是否一致，标了 `readOnlyHint=True` 的 handler 照样能写盘，一致性由工具作者自行保证。声明失实的代价也在这里：谎报只读的工具会被并行调用，写操作就可能并发执行。


### 10.4 `tools` vs `allowed_tools`：可用性层 vs 权限层

| 选项 | 作用层 | 效果 |
|---|---|---|
| `tools=["Read", "Grep"]` | 可用性 | 只有列出的**内置** tool 进 context；MCP tool 不受影响 |
| `tools=[]` | 可用性 | 移除全部内置 tool，Claude 只剩你的 MCP tool |
| `allowed_tools=[...]` | 权限 | 列出的免审批；未列出的仍可用，走权限流程 |
| `disallowed_tools=["Bash"]`（裸名） | 可用性 | 从 context 移除，Claude 不会尝试 |
| `disallowed_tools=["Bash(rm *)"]`（带范围） | 权限 | tool 仍可见，只 deny 匹配调用（Claude 可能浪费一轮去试） |

两层组合起来就是"纯业务工具 agent"的标准写法（接下方 demo 的 `server`）：

```python
options = ClaudeAgentOptions(
    mcp_servers={"demo": server},
    tools=[],                        # 可用性层：内置 tool 全部移出 context
    allowed_tools=["mcp__demo__*"],  # 权限层：仅剩的 MCP tool 免审批
)
# 效果：Claude 的工具清单里只有 demo server 的三个工具，Read/Bash 等"不存在"
# （而不是"存在但会被拒"）——这就是可用性层与权限层的差别
```


### 10.5 错误处理铁律

| handler 行为 | 后果 |
|---|---|
| 抛出未捕获异常 | **整个 agent loop 终止**，`query()` 调用直接失败，Claude 看不到错误 |
| 捕获后返回 `{"content": [...], "is_error": True}` | 循环继续，Claude 把错误当数据，可重试/换工具/解释失败 |

`is_error` 的适用范围不限于异常——业务层失败同样要标：HTTP 返回非 200、遇到不支持的输入（比如单位换算器碰到没有的转换对），都返回 `is_error: True` 而不是当正常结果。它的作用是把这条结果标成"失败的调用"，而不是让 Claude 面对一段"长得奇怪的数据"自己猜。

### 10.6 工具--除文本外的四种返回 block

`content` 数组共接受五种 block：`text` / `image` / `audio` / `resource` / `resource_link`，同一次返回可以混用；block 形态就是 MCP `CallToolResult` 的定义。

**image** —— block 里没有 URL 字段，图片字节只能 base64 内联；要返回一张网上的图，得在 handler 里自己下载字节再编码。Claude 把它当视觉输入处理（官方示例）：


In [ ]:
import base64
import httpx


@tool("fetch_image", "Fetch an image from a URL and return it to Claude", {"url": str})
async def fetch_image(args):
    async with httpx.AsyncClient() as client:
        response = await client.get(args["url"])  # 图片字节自己拿到手
    return {
        "content": [
            {
                "type": "image",
                "data": base64.b64encode(
                    response.content
                ).decode(),  # 纯 base64，无 "data:" 前缀
                "mimeType": response.headers.get("content-type", "image/png"),  # 必填
            }
        ]
    }

**resource** —— `uri` 只是给 Claude 引用的标签，SDK 不会去读那个路径；真正的内容必须内联在 `text`（文本）或 `blob`（base64 二进制）里，二选一不同给，`mimeType` 可选。适合工具产出"日后值得按名字指认"的东西（生成的文件、外部系统的一条记录）：

```python
return {"content": [{
    "type": "resource",
    "resource": {
        "uri": "file:///tmp/report.md",  # 引用标签，不是 SDK 会读的路径
        "mimeType": "text/markdown",
        "text": "# Report\n...",         # 内容在这里内联
    },
}]}
```

**audio** —— `{"type": "audio", "data": ..., "mimeType": ...}`：SDK 落盘保存，Claude 收到的是"文件路径"文本。

**resource_link** —— `{"type": "resource_link", "name": ..., "uri": ..., "description": ...}`：转换成"名字 + URI + 描述"文本。

### 10.7 `structuredContent`：机器可读结果

handler `return` 的对象 Claude 并不直接可见：SDK 收到返回值后，要把它组装成 tool result 消息发回给 Claude。`structuredContent` 是返回值里与 `content` 平级的可选 JSON 对象，作用是让 Claude 直接拿到精确字段（数值、列表），而不是从 `content` 的文本里自己解析；它影响的正是上面那步组装：

| 返回值 | Claude 实际收到的 tool result |
|---|---|
| 只有 `content` | `content` 里所有 block 原样转发 |
| `content` + `structuredContent` | 这份 JSON + `content` 里的 image/resource 等非文本 block；**text block 被丢弃** |

丢 text block 的理由：SDK 认定文本与 JSON 是同一份数据的两种写法，只保留机器可读的那份。坑也在这里——text block 里若写了 JSON 之外的额外信息，设了 `structuredContent` 后会悄悄丢失。

典型用法是同一 handler 里 image block 出图、`structuredContent` 给图背后的数据点，两者各自转发、互不重复：

```typescript
// TypeScript（Python 进程内 server 不支持，见下方 note）
return {
  content: [
    { type: "image", data: chartPngBuffer.toString("base64"), mimeType: "image/png" },
  ],
  structuredContent: {
    series: "temperature_2m",
    unit: "fahrenheit",
    points: [62.1, 63.4, 65.0, 64.2],
  },
};
```

> [!note] Python 限制
> Python 进程内 server 不支持 `structuredContent`——`@tool` 只转发 `content` 和 `is_error`，需要它就得跑独立 MCP server（§12「外部 MCP：接入现成生态」）。

In [ ]:
from claude_agent_sdk import (
    tool,
    create_sdk_mcp_server,
    ClaudeAgentOptions,
    ClaudeSDKClient,
    AssistantMessage,
    ToolUseBlock,
    ResultMessage,
)


@tool("greet", "Greet a user", {"name": str})
async def greet_user(args):
    return {"content": [{"type": "text", "text": f"Hello, {args['name']}!"}]}


@tool("add", "Add two numbers", {"a": float, "b": float})
async def add(args):
    return {"content": [{"type": "text", "text": f"Sum: {args['a'] + args['b']}"}]}


# 错误处理铁律的落地：捕获异常并返回 is_error，循环不中断，Claude 能看到并应对
@tool("divide", "Divide a by b", {"a": float, "b": float})
async def divide(args):
    try:
        return {
            "content": [{"type": "text", "text": f"Quotient: {args['a'] / args['b']}"}]
        }
    except ZeroDivisionError:
        return {
            "content": [{"type": "text", "text": "Error: division by zero"}],
            "is_error": True,
        }


server = create_sdk_mcp_server(
    name="demo", version="1.0.0", tools=[greet_user, add, divide]
)


async def demo_custom_tool():
    options = ClaudeAgentOptions(
        mcp_servers={"demo": server},
        allowed_tools=["mcp__demo__*"],  # 通配符批准 demo server 的全部工具
        max_turns=3,
    )
    async with ClaudeSDKClient(options=options) as client:
        await client.query("Greet Liangzhu, then add 2 and 3, then divide 1 by 0.")
        async for m in client.receive_response():
            # 打印每次工具调用，能看出 Claude 是真调了工具还是凭自身知识回答
            if isinstance(m, AssistantMessage):
                for block in m.content:
                    if isinstance(block, ToolUseBlock):
                        print(f"[tool call] {block.name}({block.input})")
            elif isinstance(m, ResultMessage):
                print(m.result)


await demo_custom_tool()